Plan:
- K-Fold Target Encoding - przenosimy na test przez transform
- rozważenie binningu
- interakcje z modelu gradient
- Tworzenie cech, czyszczenie
- permutation importance
- forward feature selection od najlepszego na CV
- autogluon (multilayer stacking)


In [38]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import TargetEncoder

In [43]:
X_train = pd.read_csv('data/train.csv')
X_test = pd.read_csv('data/test.csv')
y_train = X_train.pop('Churn') 

In [44]:
X_train = X_train.drop(columns=['id'])
X_test = X_test.drop(columns=['id'])

#### Używanie podzbioru dla szybszych obliczeń - sprawdzamy dodanie których kolumn (interakcji/nowych cech) najbardziej zwiększa roc auc

In [45]:
X_sample, _, y_sample, _ = train_test_split(
    X_train, 
    y_train, 
    train_size=0.10,   
    random_state=42,   
    stratify=y_train   
)

In [46]:
y_sample = y_sample.map({'No': 0, 'Yes': 1})
categorical_cols = X_sample.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
if 'SeniorCitizen' in X_sample.columns and 'SeniorCitizen' not in categorical_cols:
    categorical_cols.append('SeniorCitizen')
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

In [47]:
encoder = TargetEncoder(cv=5, random_state=42)
X_encoded = X_sample.copy()
X_encoded[categorical_cols] = encoder.fit_transform(X_sample[categorical_cols], y_sample)
df = X_encoded.copy()
df['Churn'] = y_sample.values if isinstance(y_sample, pd.Series) else y_sample

In [48]:
df['tenure_safe'] = df['tenure'].replace(0, 0.001)
avg_historical_charge = df['TotalCharges'] / df['tenure_safe']
df['Int_PriceShockRatio'] = df['MonthlyCharges'] / (avg_historical_charge + 0.001)
df['Int_CostPerTenure'] = df['MonthlyCharges'] / df['tenure_safe']
df['Int_ImpliedTenureDiff'] = (df['TotalCharges'] / (df['MonthlyCharges'] + 0.001)) - df['tenure']
df['Int_CumulativeBurden'] = df['TotalCharges'] * df['MonthlyCharges']

In [49]:
def evaluate_features(features_list, df, target='Churn'):
    X = df[features_list]
    y = df[target]
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    model = RandomForestClassifier(n_estimators=50, max_depth=6, random_state=42, n_jobs=-1)
    
    scores = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
    return np.mean(scores)

In [50]:
base_features = categorical_cols + numeric_cols
interactions = ['Int_PriceShockRatio', 'Int_CostPerTenure', 'Int_ImpliedTenureDiff', 'Int_CumulativeBurden']

results = {}

base_roc_auc = evaluate_features(base_features, df)
results['Baseline (Wszystkie surowe)'] = base_roc_auc
print(f"ROC AUC - Baseline (kategorie + numeryczne): {base_roc_auc:.4f}")

for inter in interactions:
    current_features = base_features + [inter]
    roc_auc = evaluate_features(current_features, df)
    results[f'Baseline + {inter}'] = roc_auc
    improvement = roc_auc - base_roc_auc
    print(f"ROC AUC - dodano '{inter}': {roc_auc:.4f} (Różnica: {improvement:+.4f})")

all_features = base_features + interactions
all_roc_auc = evaluate_features(all_features, df)
results['Wszystkie zmienne (z interakcjami)'] = all_roc_auc
print(f"ROC AUC - Wszystkie zmienne: {all_roc_auc:.4f} (Różnica: {all_roc_auc - base_roc_auc:+.4f})")

print("\n--- Top 10 Feature Importance dla pełnego modelu ---")
model_all = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42).fit(df[all_features], df['Churn'])
importances = pd.Series(model_all.feature_importances_, index=all_features).sort_values(ascending=False)
print(importances.head(10))

ROC AUC - Baseline (kategorie + numeryczne): 0.9073
ROC AUC - dodano 'Int_PriceShockRatio': 0.9077 (Różnica: +0.0004)
ROC AUC - dodano 'Int_CostPerTenure': 0.9080 (Różnica: +0.0007)
ROC AUC - dodano 'Int_ImpliedTenureDiff': 0.9075 (Różnica: +0.0002)
ROC AUC - dodano 'Int_CumulativeBurden': 0.9075 (Różnica: +0.0002)
ROC AUC - Wszystkie zmienne: 0.9075 (Różnica: +0.0002)

--- Top 10 Feature Importance dla pełnego modelu ---
Int_CostPerTenure       0.207443
PaymentMethod           0.152511
Contract                0.127010
InternetService         0.110320
tenure                  0.079479
OnlineSecurity          0.061530
TechSupport             0.050956
MonthlyCharges          0.047484
TotalCharges            0.038653
Int_CumulativeBurden    0.030283
dtype: float64


##### Można zauważyć, że CostPerTenure jest silnym predyktorem, CumulativeBurden i PriceShockRatio tez wygladają nieźle

### Sprawdzenie interakcji cech kategorycznych

#### Ad. 1) Czy dla Streaming Movies/Tv wystarczy dla polepszenia predykcji sama informacja czy nie było internet service

In [53]:
cross_tab = pd.crosstab(X_train['InternetService'], X_train['StreamingTV'])
cross_tab

StreamingTV,No,No internet service,Yes
InternetService,,,
DSL,104608,0,76473
Fiber optic,108558,0,163828
No,0,140727,0


#### Widać, że cała wiedza zawarta w StreamingTV jest zawarta w InternetService - można usunąć predyktory StreamingTV i StreamingMovies